In [49]:
def triangleFinite(w,m1,m2,m3):
    if (w^m1).is_one() or (w^m2).is_one() or (w^m3).is_one():
        return True
    else:
        return False

In [50]:
#checks is r1 seperates r2 from the identity, a wall seperates itself from the identity
def seperateFromOne(r1,r2,m1,m2,m3):
    r=r1*r2
    if (r1*(r2^-1)).is_one():
        return True
    if triangleFinite(r,m1,m2,m3):
        return False
    elif (r).length()<(r2).length():
        return True
    else:
        return False

In [51]:
def wallsOfOne(G,m1,m2,m3):
    #fin is the indicator for the process of finding walls, parallel is a statement about the current wall being checked
    fin=False
    parallel=False
    #Initializing the list with the walls of the fundamental chamber, so start check at walls gsg^-1 where g has length 1
    n=1
    s=G.simple_reflections()
    H=[]
    for t in s:
        H.append(tuple(t.reduced_word()))
    while fin==False:
        #if we find any new walls we stop, if we find a new wall we flip this
        fin=True
        #looking at walls at distance n from the identity
        for g in list(G.elements_of_length(n)):
            #got 3 reflections around any point, redcing this to 2 is possible
            for t in s:
                #we assume the new wall is not seperated from 1 until we find a wall in H that does
                parallel=False
                #the reflection we are checking
                gRef=g*t*(g^-1)
                #print(gRef.reduced_word())
                #checking if it is a new one
                for rl in H:
                    #convert the reduced word to group element
                    r=G.from_reduced_word(rl)
                    #check if g is seperated from the unit by this wall, if it is seperated we can ignore it
                    if seperateFromOne(r,gRef,m1,m2,m3):
                            parallel=True
                            #print("seperated by ")
                            #print(rl)
                            #print(" ")
                #if we found a new wall add (a reduced word for it) to the list
                #also need to look a layer deeper
                if parallel==False:
                    H.append(tuple(gRef.reduced_word()))
                    fin=False
        #increase the search radius
        n=n+1
    return H, n
                

In [52]:
#reflections seerating an element from the unit bring it closer
def seperateFromOneElem(r,g):
    if (r*g).length()<g.length():
        return True
    else:
        return False

In [53]:
#find all the walls seperating g not seperated from g by another wall
def wallsOfElem(G,g,m1,m2,m3):
    #all walls seperating g from e are represented in (any) shortest representative
    l=g.length()
    w=g.reduced_word()
    #closest one is definitely one of them
    h=G.from_reduced_word(w[:l-1])
    r=G.from_reduced_word([w[l-1]])
    ref=h*r*(h^-1)
    Wg=[]
    Wg.append(tuple(ref.reduced_word()))
    #work down, knowing the radius of the walls seperating the identity gives a bound, might be n-1, but I want to be safe
    for i in range(l):
        #check if seperated by one closer
        parallel=False
        #underflow issues
        k=max(l-i-1,0)
        #build reflection
        h=G.from_reduced_word(w[:k])
        r=G.from_reduced_word([w[k]])
        ref=h*r*(h^-1)
        #check if it is a new one
        for rl in Wg:
            r2=G.from_reduced_word(rl)
            #check if we should not add
            if seperateFromOne(ref,r2,m1,m2,m3):
                parallel=True
        if parallel==False:
            Wg.append(tuple(ref.reduced_word()))
    return Wg
        

In [54]:
def voraciousProj(G,g,m1,m2,m3):
    Wg=wallsOfElem(G,g,m1,m2,m3)
    lmin=len(Wg)
    #one of the geodesics 
    Can=g.reduced_words()
    l=g.length()
    #the voracious projection is passed by all geodesics, so after picking one we just go back until we pass all the walls
    #pass a minimum of |Wg| of them 
    for i in range(lmin-1,l+1):
        CanTemp=[]
        for h in Can:
            CanTemp.append(h[:l-i])
        for p in CanTemp:
            seperate=True
            for rl in Wg:
                r=G.from_reduced_word(rl)
                if seperateFromOneElem(r,G.from_reduced_word(p)):
                    seperate=False
            if seperate==True:
                return G.from_reduced_word(p)
        Can=CanTemp
    return g

In [55]:
def actionElemOnWalls(G,g,Wg):
    gInvWg=[]
    for rl in Wg:
        r=W.from_reduced_word(rl)
        gInvWg.append(tuple(((g)*r*(g^-1)).reduced_word()))
    return gInvWg

In [56]:
from sage.combinat.finite_state_machine import FSMState
def setupVorFSA(U):
    States=[]
    P=list(subsets(U))
    for u in P:
        States.append(FSMState(tuple(u),is_final=True))
    VorFSA=Automaton()
    VorFSA.add_states(States)
    return VorFSA

In [63]:
def vorProjOne(G,m1,m2,m3):
    projElem=[]
    done=False
    l=1
    while not done:
        done=True
        tempCand=G.elements_of_length(l)
        l=l+1
        for g in tempCand:
            if voraciousProj(G,g,m1,m2,m3).is_one():
                projElem.append(g)
                done=False
    return projElem

In [64]:
M=matrix([[1,3,7],[3,1,3],[7,3,1]])

In [65]:
W=CoxeterGroup(M)

In [66]:
U, n =wallsOfOne(W,3,3,7)

In [67]:
voraciousProj(W,W.from_reduced_word([1,3,1,3,1,3,1]),3,3,7).reduced_word()

[]

In [68]:
Proj=vorProjOne(W,3,3,7)
for g in Proj:
    print(g.reduced_word())

[1]
[2]
[3]
[1, 2]
[1, 3]
[2, 1]
[2, 3]
[3, 1]
[3, 2]
[1, 2, 1]
[1, 3, 1]
[2, 3, 2]
[3, 1, 3]
[1, 2, 1, 3]
[1, 3, 1, 3]
[2, 3, 2, 1]
[3, 1, 3, 1]
[1, 3, 1, 3, 1]
[3, 1, 3, 1, 3]
[1, 3, 1, 3, 1, 3]
[3, 1, 3, 1, 3, 1]
[1, 3, 1, 3, 1, 3, 1]
[1, 3, 1, 3, 1, 3, 1, 2]


In [15]:
VorFSA=setupVorFSA(U)

In [16]:
VorFSA

Automaton with 1024 states